# Experiment 4: HOG + GLCM + SVM

**Research Question:** Does combining shape and texture improve classification?

In this experiment, we extract both the **HOG** (shape) and **GLCM** (texture) features for each image, concatenate them into a single comprehensive feature vector, and train an **SVM** classifier. We also use a standard scaler to ensure all feature values are on a comparable scale, which is critical for SVMs.

# Step 0: Google Colab Setup

In [ ]:
import os
try:
    from google.colab import drive
    drive.mount('/content/drive')
    print("\nGoogle Drive Mounted successfully!")
except ImportError:
    print("Not running in Google Colab. Skipping Drive mount.")

# Step 1: Imports and Basics

In [ ]:
import os
import cv2
import numpy as np
from skimage.feature import hog, graycomatrix, graycoprops
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    classification_report, accuracy_score,
    precision_score, recall_score, f1_score, confusion_matrix
)
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.notebook import tqdm

LABEL_MAP = {'glioma': 0, 'meningioma': 1, 'notumor': 2, 'pituitary': 3}
CLASS_NAMES = list(LABEL_MAP.keys())

# Step 2: Extract Combined Features
To iterate through all images, compute both HOG and GLCM, and concatenate the arrays.

In [ ]:
def extract_hog_glcm(image):
    # HOG Extraction
    hog_features = hog(
        image,
        orientations=9,
        pixels_per_cell=(8, 8),
        cells_per_block=(2, 2),
        block_norm='L2-Hys',
        visualize=False
    )
    
    # GLCM Extraction
    img_uint8 = image.astype(np.uint8) if image.dtype != np.uint8 else image
    glcm = graycomatrix(img_uint8, distances=[1], angles=[0, np.pi/4, np.pi/2, 3*np.pi/4], levels=256, symmetric=True, normed=True)
    properties = ['contrast', 'dissimilarity', 'homogeneity', 'energy', 'correlation', 'ASM']
    glcm_features = []
    for prop in properties:
        glcm_features.extend(graycoprops(glcm, prop).flatten())
    
    # Combine
    combined = np.hstack((hog_features, glcm_features))
    return combined

def load_and_extract_combined(base_path, target_size=(128, 128)):
    X, y = [], []
    if not os.path.exists(base_path):
        print(f"ERROR: The path {base_path} does not exist! Please check your base_dir variable.")
        return np.array(X), np.array(y)

    print(f"Loading and extracting combined features from {base_path}...")
    for class_name, label_idx in tqdm(LABEL_MAP.items(), desc="Classes"):
        class_folder = os.path.join(base_path, class_name)
        if not os.path.exists(class_folder):
            continue

        for filename in os.listdir(class_folder):
            if filename.lower().endswith(('.jpg', '.jpeg', '.png')):
                img_path = os.path.join(class_folder, filename)
                img = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)
                if img is not None:
                    img_resized = cv2.resize(img, target_size)
                    combined_features = extract_hog_glcm(img_resized)
                    X.append(combined_features)
                    y.append(label_idx)
    return np.array(X), np.array(y)

base_dir = '/content/drive/MyDrive/NeuroScan'

# --- Auto-detect for local execution ---
if 'base_dir' not in locals():
    current_dir = os.getcwd()
    base_dir = current_dir if os.path.exists(os.path.join(current_dir, 'data', 'Training')) else os.path.dirname(current_dir)

train_dir = os.path.join(base_dir, 'data', 'Training')
test_dir  = os.path.join(base_dir, 'data', 'Testing')

print("Loading Training Data...")
X_train, y_train = load_and_extract_combined(train_dir)

print("Loading Testing Data...")
X_test, y_test = load_and_extract_combined(test_dir)

if len(X_train) > 0:
    print(f"\nTraining Combined feature shape: {X_train.shape}")
    print(f"Testing Combined feature shape:  {X_test.shape}")
else:
    print("\nFailed to load images. Please fix the base_dir path above.")

# Step 3: Standardize Features and Train SVM
Because HOG values are normalized decimals and GLCM properties have widely varying scales, we must scale the data before fitting the SVM.

In [ ]:
print("Standardizing features...")
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("Training SVM on Combined features...")
model = SVC(kernel='rbf', C=10, gamma='scale', random_state=42)
model.fit(X_train_scaled, y_train)
print("Training Complete")

# Step 4: Evaluation

In [ ]:
y_pred = model.predict(X_test_scaled)

accuracy  = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred, average='weighted')
recall    = recall_score(y_test, y_pred, average='weighted')
f1        = f1_score(y_test, y_pred, average='weighted')

print("================ EVALUATION METRICS ================")
print(f"Accuracy  : {accuracy:.4f}")
print(f"Precision : {precision:.4f}")
print(f"Recall    : {recall:.4f}")
print(f"F1-Score  : {f1:.4f}")
print("====================================================")

print("\nClassification Report:")
print(classification_report(y_test, y_pred, target_names=CLASS_NAMES))

# Confusion Matrix
cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Purples',
            xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES)
plt.title('Confusion Matrix: HOG+GLCM using SVM')
plt.ylabel('True Label')
plt.xlabel('Predicted Label')
plt.tight_layout()
plt.show()